<a href="https://colab.research.google.com/github/nadynstary/Astria-s-Song/blob/main/Sonification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
from PIL import Image
from scipy.io import wavfile
from IPython.display import Audio, display

SR = 22050
SCALE = [0, 2, 4, 7, 9, 11]  # C D E G A B

def bell_note(freq, dur, vel=1.0):
    t = np.arange(int(dur * SR)) / SR
    tone = (np.sin(2*np.pi*freq*t)
            + 0.25*np.sin(2*np.pi*2*freq*t) * np.exp(-t*3)
            + 0.08*np.sin(2*np.pi*3*freq*t) * np.exp(-t*5))
    env = np.minimum(1, t/0.004) * np.exp(-t*1.6)
    return tone * env * vel

def sonify_radial(img, duration=30, n_steps=120, notes_per_step=3, threshold=0.18,
                  low_note=55, high_note=91, center=(0.5, 0.5), outside_in=True,
                  invert=False, tail=3.0, n_angles=360):
    g = img.convert("L")
    g.thumbnail((500, 500))
    a = np.asarray(g, dtype=float) / 255.0
    if invert:
        a = 1.0 - a
    H, W = a.shape
    cx, cy = center[0] * (W - 1), center[1] * (H - 1)
    r_max = max(np.hypot(cx, cy), np.hypot(W - 1 - cx, cy),
                np.hypot(cx, H - 1 - cy), np.hypot(W - 1 - cx, H - 1 - cy))
    r_max *= 0.75

    midis = [m for m in range(low_note, high_note + 1) if m % 12 in SCALE]
    freqs = [440 * 2 ** ((m - 69) / 12) for m in midis]
    angles = np.linspace(0, 2*np.pi, n_angles, endpoint=False)

    step_t = duration / n_steps
    out = np.zeros((int((duration + tail) * SR), 2))

    for j in range(n_steps):
        prog = j / n_steps
        r = r_max * (1 - prog) if outside_in else r_max * prog
        r = max(r, 2)
        xs = np.round(cx + r*np.cos(angles)).astype(int)
        ys = np.round(cy + r*np.sin(angles)).astype(int)
        ok = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
        b = np.zeros(n_angles)
        b[ok] = a[ys[ok], xs[ok]]
        b = np.convolve(np.r_[b[-1], b, b[0]], np.ones(3)/3, mode="valid")

        peaks = (b > np.roll(b, 1)) & (b >= np.roll(b, -1)) & (b > threshold)
        idxs = np.where(peaks)[0]
        idxs = idxs[np.argsort(b[idxs])[-notes_per_step:]]

        frac = r / r_max                              # 1 = edge, 0 = center
        n_idx = int(round(frac * (len(freqs) - 1)))   # edge = high pitch
        closeness = 1 - frac if outside_in else frac
        gain = 0.12 + 0.88 * closeness ** 1.5         # louder near the center

        for k in idxs:
            pan = np.cos(angles[k])                   # left <-> right by angle
            th = (pan + 1) * np.pi / 4
            note = bell_note(freqs[n_idx], 2.5, vel=b[k] * gain)
            s = int(j * step_t * SR)
            e = min(s + len(note), len(out))
            out[s:e, 0] += note[:e - s] * np.cos(th)
            out[s:e, 1] += note[:e - s] * np.sin(th)

    for d, gdb in ((int(0.23*SR), 0.30), (int(0.37*SR), 0.20)):  # echo
        out[d:] += gdb * out[:-d].copy()
    return out / (np.max(np.abs(out)) + 1e-9) * 0.9



In [5]:
# ===================== NEW: sparkle + echo =====================
from scipy.signal import butter, sosfilt, fftconvolve, find_peaks

def _filt(x, kind, fc, order=4):
    sos = butter(order, fc, kind, fs=SR, output="sos")
    return sosfilt(sos, x)

def _rms(x):
    return np.sqrt(np.mean(x ** 2)) + 1e-9

def _bell(freq, dur=1.6):
    t = np.arange(int(dur * SR)) / SR
    y = np.zeros_like(t)
    for ratio, amp, decay in [(1, 1.0, 3.0), (2.76, 0.5, 5.0), (5.4, 0.3, 8.0), (8.93, 0.15, 12.0)]:
        if freq * ratio < SR / 2 * 0.9:
            y += amp * np.sin(2 * np.pi * freq * ratio * t) * np.exp(-t * decay)
    return y * np.minimum(t / 0.002, 1)

def _taps(x, delay, fb, n_taps=9):
    out = np.zeros_like(x)
    d = int(delay * SR)
    for k in range(1, n_taps + 1):
        s = k * d
        if s >= len(x):
            break
        out[s:] += (fb ** k) * x[:-s]
    return out

def sparkle_and_echo(audio, sparkle=0.7, echo=0.6, reverb=0.8, tail=5.0, seed=1):
    """Keeps the original sound untouched (dry) and adds on top:
       1) twinkle  : the high frequencies flicker like stars
       2) glints   : tiny bells triggered whenever the sound swells
       3) echo     : stereo ping-pong delay
       4) reverb   : long shimmering tail
    """
    rng = np.random.default_rng(seed)
    dry = np.asarray(audio, float)
    if dry.ndim == 2:
        dry = dry.mean(axis=1)
    n = len(dry) + int(tail * SR)          # extra room so the echo can ring out
    dry = np.pad(dry, (0, n - len(dry)))
    ref = _rms(dry[: len(dry) - int(tail * SR)])

    # 1) twinkle: highpassed copy x random flicker
    hi = _filt(dry, "highpass", 2500)
    flick = _filt(rng.normal(size=n), "lowpass", 18, 2)
    flick = np.clip(flick / (np.max(np.abs(flick)) + 1e-9), 0, None) ** 2
    flick /= flick.max() + 1e-9
    twinkle = hi * flick
    twinkle *= min(ref / _rms(twinkle), 20) * 0.5 * sparkle

    # 2) glints: bell notes on the peaks of the loudness curve
    env = _filt(np.abs(dry), "lowpass", 6, 2)
    peaks, props = find_peaks(env, height=0.35 * env.max(), distance=int(0.15 * SR))
    gL, gR = np.zeros(n), np.zeros(n)
    for p, h in zip(peaks, props["peak_heights"]):
        f = 1568 * 2 ** (PENTATONIC[rng.integers(5)] / 12) * rng.choice([1, 1.5, 2])
        b = _bell(f) * (h / env.max()) * 0.35 * sparkle * ref * 4
        pan = rng.uniform(-0.9, 0.9)
        e = min(n, p + len(b))
        gL[p:e] += b[: e - p] * np.cos((pan + 1) * np.pi / 4)
        gR[p:e] += b[: e - p] * np.sin((pan + 1) * np.pi / 4)

    dL = dry + twinkle + gL
    dR = dry + twinkle + gR

    # 3) echo: different delay per side = ping-pong width
    src = _filt(dL + dR, "lowpass", 5000, 2) * 0.5
    eL = _taps(src, 0.33, 0.55)
    eR = _taps(src, 0.47, 0.55)

    # 4) reverb: long decaying noise, kept bright so highs shimmer
    def ir(seed_):
        r = np.random.default_rng(seed_)
        m = int(4.0 * SR)
        t = np.arange(m) / SR
        h = r.normal(size=m) * np.exp(-t * 1.5)
        h[: int(0.02 * SR)] *= np.linspace(0, 1, int(0.02 * SR))
        return h
    wet_in = dL + dR + echo * (eL + eR)
    rL = fftconvolve(wet_in, ir(11))[:n]
    rR = fftconvolve(wet_in, ir(12))[:n]
    rL *= ref / _rms(rL) * 0.8 * reverb
    rR *= ref / _rms(rR) * 0.8 * reverb

    L = dL + echo * eL + rL
    R = dR + echo * eR + rR
    L *= np.minimum(np.arange(n)[::-1] / (SR * 1.5), 1)   # gentle fade-out
    R *= np.minimum(np.arange(n)[::-1] / (SR * 1.5), 1)
    return normalize(np.stack([L, R], axis=1))


In [8]:
from google.colab import files
from IPython.display import Audio, display
import numpy as np
from PIL import Image
from scipy.io import wavfile

# Define the missing variables needed by sparkle_and_echo
globals()['PENTATONIC'] = [0, 2, 4, 7, 9, 11]
globals()['normalize'] = lambda x: x / (np.max(np.abs(x)) + 1e-9)

uploaded = files.upload()
print("uploaded:", list(uploaded.keys()))

path = list(uploaded.keys())[0]
img = Image.open(path)
print("image size:", img.size)

audio = sonify_radial(img, duration=10)
audio = sparkle_and_echo(audio)
print("audio samples:", len(audio), "| max:", float(np.max(np.abs(audio))))

wavfile.write("scan.wav", SR, (audio * 32767).astype(np.int16))
display(Audio("scan.wav"))

Saving aurora.png to aurora (3).png
uploaded: ['aurora (3).png']
image size: (512, 512)
audio samples: 396900 | max: 0.999999999015746


In [ ]:
audio = sonify_radial(img)   # takes ~10-20 seconds to compute
audio = sparkle_and_echo(audio)
wavfile.write("radial.wav", SR, (audio * 32767).astype(np.int16))
display(Audio("radial.wav"))